In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(12046)

In [2]:
sequence_len = 64
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [5]:
# softmax对于-inf的计算结果
a = torch.tensor([1, 2, float('-inf')]).float()
F.softmax(a)

C:\Users\admin\AppData\Local\Temp\ipykernel_14600\3829447422.py:3: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  F.softmax(a)


tensor([0.2689, 0.7311, 0.0000])

In [6]:
# K:(B,T,H)
# Q:(B,T,H)
# k @ Q.transpose(-2,-1):(B,T,T)
# 内积就相当于计算两个矩阵向量相乘
scores = torch.randn(1, 4, 4)
scores

tensor([[[ 1.0185, -1.3091,  1.2908,  0.5276],
         [-0.2985,  1.6259,  2.0433, -0.6417],
         [ 0.8795, -1.0512,  1.1491,  0.6116],
         [ 0.2128, -0.5512,  0.0450,  0.5010]]])

In [7]:
#定义下三角矩阵
tril = torch.tril(torch.ones(4, 4))
s = scores.masked_fill(tril == 0, float('-inf'))
s

tensor([[[ 1.0185,    -inf,    -inf,    -inf],
         [-0.2985,  1.6259,    -inf,    -inf],
         [ 0.8795, -1.0512,  1.1491,    -inf],
         [ 0.2128, -0.5512,  0.0450,  0.5010]]])

In [10]:
# 定义权重分布
F.softmax(s, dim=-1)

tensor([[[1.0000, 0.0000, 0.0000, 0.0000],
         [0.1274, 0.8726, 0.0000, 0.0000],
         [0.4074, 0.0591, 0.5335, 0.0000],
         [0.2743, 0.1278, 0.2319, 0.3659]]])

In [11]:
# softmax 对方差的敏感性
x = torch.randn(1, 8)
x.std(), F.softmax(x, dim=-1)

(tensor(1.0912),
 tensor([[0.1236, 0.0400, 0.0791, 0.0175, 0.6045, 0.0262, 0.0633, 0.0457]]))

In [12]:
F.softmax(1000 * x, dim=-1)

tensor([[0., 0., 0., 0., 1., 0., 0., 0.]])

因此我们需要保证，我们输入给softmax函数的这个输入它的方差要是等于1的

In [13]:
# 对其分数的方差变化
B, T, H = 32, 100, 10
K = torch.randn(B, T, H)
Q = torch.randn(B, T, H)
scores = K @ Q.transpose(-2, -1) / H ** 0.5 # 归一化处理
scores.std()

tensor(1.0065)

In [14]:
def attention(query, key, value, dropout, mask=None):
    #query,key,value:(B,T,H)
    # mask:          (T,T)
    # output:        (B,T,H)
    B, T, H = query.shape
    scores = query @ key.transpose(-2, -1) / H ** 0.5
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    w_att = F.softmax(scores, dim=-1)  # (B,T,T)
    out = w_att @ value                # (B,T,H)
    return out

In [16]:
class MaskedAttention(nn.Module):
    # 单项自注意力

    def __init__(self, emb_size, head_size):
        # emb_size:C,head_size:H
        # emb_size表示输入的长度
        super().__init__()
        self.key = nn.Linear(emb_size, head_size, bias=False) # 不需要截距项的原因是因为在神经网络中有一个残差连接的方式
        self.query = nn.Linear(emb_size, head_size, bias=False)
        self.value = nn.Linear(emb_size, head_size, bias=False)
        # 定义下三角矩阵
        self.register_buffer('tril', torch.tril(torch.ones(sequence_len, sequence_len)))
        self.dp = nn.Dropout(0.4)

    def forward(self, x):
        #x :(B,T,C)
        # out:(B,T,H)
        B, T, C = x.shape
        k = self.key(x)  # (B,T,H)
        q = self.query(x)  #(B,T,H)
        v = self.value(x)  # (B,T,H)
        mask = self.tril[:T, :T]
        out = attention(q, k, v, self.dp, mask)
        return out

In [17]:
m = MaskedAttention(3, 4)
x = torch.randn(5, 10, 3)
m(x).shape

torch.Size([5, 10, 4])